# Quantifying Shared-Design Heritage in a Power Transformer Fleet with PROC INBREED

## Executive Summary

A grid operator's substation transformers are engineered in successive design generations, each new model derived from two predecessor designs. This notebook treats that engineering genealogy as a pedigree and uses **PROC INBREED** to compute inbreeding and additive-relationship (covariance) coefficients, quantifying how much design heritage any two assets share.

The pedigree is built so the structure is interpretable: two of the four current-fleet models descend from a pair of **sibling** predecessor designs and therefore carry concentrated heritage, while the others draw on disjoint lineages. The procedure recovers this exactly. The two sibling-derived models (`G3_T1`, `G3_T3`) each carry an inbreeding coefficient of **F = 0.25**; the other two (`G3_T2`, `G3_T4`) are **F = 0**. The fleet's average inbreeding coefficient is **0.0417**. Reading the additive-relationship matrix, the least-related current-fleet pair is **`G3_T1` & `G3_T3` (relationship = 0)** — they share no ancestry and make the strongest redundant (N-1) pairing, which matters because `G3_T1` is itself one of the most inbred designs.

## Data Sources

| Dataset | Rows | Key Variables | Description |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | A small, deterministic engineering pedigree of substation transformer designs across three non-overlapping design generations (4 founding platforms, 4 second-generation derivatives, 4 current-fleet models). Each non-founder design records the two distinct predecessor designs (`Parent1`, `Parent2`) it was derived from. `Sex` tags the lead-design role (M = mechanical/core lineage, F = electrical/winding lineage). The pedigree is hand-specified — not random — so the relatedness structure is known in advance and can be checked against the procedure's output. |

# Quantifying Shared-Design Heritage in a Power Transformer Fleet

## Why a utility cares about "inbreeding"

A transmission and distribution operator runs hundreds of substation power transformers. To control engineering cost and qualification effort, manufacturers rarely design each transformer from scratch — every new model **inherits** core geometry, winding topology, insulation systems, and bushing designs from one or two predecessor models. Over several procurement cycles this produces a genuine *engineering genealogy*: a pedigree of designs.

That shared heritage is a hidden reliability risk. If two transformers that protect the same load (a redundant **N-1** pair) descend from the same ancestral design, a latent design defect — a resonance mode, an insulation-aging mechanism, a bushing flashover pathway — is likely to be present in **both** units. A single root cause can then take out the redundant pair simultaneously: a *common-mode failure*.

**PROC INBREED** was built to quantify exactly this kind of shared ancestry. Although its origins are in animal and plant breeding, its mathematics — Wright's additive-relationship recursion — is domain-agnostic: it measures the fraction of design heritage two individuals share through common ancestors. We map the genetics vocabulary onto asset engineering:

| INBREED concept | Utility interpretation |
|---|---|
| Individual | A transformer design / model |
| Parent (sire/dam) | A predecessor design it was derived from |
| Generation (CLASS) | A procurement / design cycle |
| Inbreeding coefficient *F* | Degree of self-similar heritage within a design |
| Covariance / relationship coefficient | Shared-design heritage between two designs |
| Least-related pair | Best redundant pairing for N-1 resilience |

## Step 1 — Specify the design pedigree

We enter `DesignLineage` directly so the relatedness structure is unambiguous. There are three **non-overlapping design generations**:

- **Generation 1** — four founding platform designs (`G1_P1`-`G1_P4`) with no predecessors (blank parents).
- **Generation 2** — four derivative designs (`G2_D1`-`G2_D4`), each engineered from two **distinct** Generation-1 platforms. `G2_D1` and `G2_D2` are *full siblings* (both from `G1_P1` x `G1_P2`); `G2_D3` and `G2_D4` are full siblings (both from `G1_P3` x `G1_P4`).
- **Generation 3** — four current-fleet models (`G3_T1`-`G3_T4`). `G3_T1` is built from the sibling pair `G2_D1` x `G2_D2`, and `G3_T3` from the sibling pair `G2_D3` x `G2_D4`; these two designs therefore concentrate heritage. `G3_T2` and `G3_T4` each cross the two disjoint lineages.

The `ID`, `Parent1`, and `Parent2` columns carry the pedigree; `Sex` records which engineering lineage led the design. A short follow-up DATA step blanks the placeholder `.` so the founding platforms have empty parents, as PROC INBREED expects.

In [1]:
data DesignLineage;
   length ID $12 Parent1 $12 Parent2 $12 Sex $1;
   infile datalines truncover;
   input Generation ID $ Parent1 $ Parent2 $ Sex $;
   datalines;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
run;

/* Founding platforms have no predecessors: blank the placeholder dots */
data DesignLineage;
   set DesignLineage;
   if Parent1 = '.' then Parent1 = ' ';
   if Parent2 = '.' then Parent2 = ' ';
run;

title 'Transformer Design Pedigree';
proc print data=DesignLineage noobs;
   var Generation ID Parent1 Parent2 Sex;
run;

                                              Transformer Design Pedigree                                               


Generation     ID  Parent1  Parent2  Sex
----------  -----  -------  -------  ---
         1  G1_P1                    M
         1  G1_P2                    M
         1  G1_P3                    M
         1  G1_P4                    F
         2  G2_D1  G1_P1    G1_P2    M
         2  G2_D2  G1_P1    G1_P2    F
         2  G2_D3  G1_P3    G1_P4    F
         2  G2_D4  G1_P3    G1_P4    M
         3  G3_T1  G2_D1    G2_D2    M
         3  G3_T2  G2_D1    G2_D3    F
         3  G3_T3  G2_D3    G2_D4    F
         3  G3_T4  G2_D2    G2_D4    M

NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 colum

## Step 2 — Inbreeding coefficients by design generation

We run PROC INBREED in **multi-generation mode** by naming `Generation` in the `CLASS` statement, which prints the pedigree summary by generation. The `VAR` statement supplies the three pedigree columns in the required order: individual, first predecessor, second predecessor.

- The **IND** option prints each design's inbreeding coefficient *F* — a measure of how concentrated its own heritage is. A design built from two closely related predecessors carries a high *F*.
- The **MATRIX** option prints the full additive-relationship matrix so we can read pairwise heritage directly.
- The **AVERAGE** option appends the fleet-wide average inbreeding coefficient — a single design-diversity summary.

Values near 0 mean independent lineages; *F* rises as a design's two predecessors become more closely related.

In [2]:
title 'Inbreeding Coefficients by Design Generation';
proc inbreed data=DesignLineage ind average matrix;
   var ID Parent1 Parent2;
   class Generation;
run;

                                      Inbreeding Coefficients by Design Generation                                      


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by GENERATION/Class
--------------------------------------
GENERATION 1            Members = 4
GENERATION 2            Members = 4
GENERATION 3            Members = 4

Inbreeding Coefficients
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
G3_T1                 0.250000
G3_T2                 0.000000
G3_T3                 0.250000
G3_T4                 0.000000
Average Inbreeding Coefficient: 0.041667


NOTE: Option TITLE changed to Inbreeding Coefficients by De

## Step 3 — Covariance coefficients saved for downstream risk scoring

Replacing the inbreeding view with the **COVAR** option reports the same relationships as **covariance (additive-relationship) coefficients**, the form an asset-management team would feed into a redundancy-risk score. The **OUTCOV=** option captures the full matrix as a dataset (`DesignCovar`), where columns `Col1`-`Col12` hold the relationship of each design to every other design (in pedigree order) — ready to join back to substation assignments.

We print the output dataset so the saved matrix is visible alongside the listing.

In [3]:
title 'Covariance (Relationship) Coefficients';
proc inbreed data=DesignLineage covar matrix outcov=DesignCovar;
   var ID Parent1 Parent2;
   class Generation;
run;

title 'OUTCOV= Relationship Matrix Saved for Risk Scoring';
proc print data=DesignCovar noobs;
run;

                                         Covariance (Relationship) Coefficients                                         


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by GENERATION/Class
--------------------------------------
GENERATION 1            Members = 4
GENERATION 2            Members = 4
GENERATION 3            Members = 4

Inbreeding Coefficients
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
G3_T1                 0.250000
G3_T2                 0.000000
G3_T3                 0.250000
G3_T4                 0.000000

Covariance Matrix (Additive Genetic Relationship)
--------------------------------------------------


## Step 4 — Least-related pairings for redundant (N-1) installations

The saved `DesignCovar` matrix is exactly what a redundancy study needs. The relationship of design *i* to design *j* lives in row *i*, column `Col`*j* (columns are in pedigree order, so `Col9`-`Col12` correspond to the four current-fleet models `G3_T1`-`G3_T4`).

We read the four current-fleet rows out of `DesignCovar`, form every unordered current-fleet pair, attach its relationship coefficient, and sort least-related first. The goal is to choose redundant pairs whose shared heritage is **lowest** — those minimize the chance that one inherited design flaw disables both halves of an N-1 protected position.

In [4]:
/* Pull the four current-fleet rows; Col9..Col12 are the relationships
   to G3_T1..G3_T4 (pedigree column order). Build each unordered pair. */
data Pairs;
   set DesignCovar;
   where ID in ('G3_T1','G3_T2','G3_T3','G3_T4');
   length DesignA $12 DesignB $12;
   array g3{4} Col9 Col10 Col11 Col12;
   array nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   do j = 1 to 4;
      DesignB = nm{j};
      if DesignA < DesignB then do;
         Relationship = g3{j};
         output;
      end;
   end;
   keep DesignA DesignB Relationship;
run;

proc sort data=Pairs; by Relationship; run;

title 'Candidate Redundant (N-1) Pairings, Least Related First';
proc print data=Pairs noobs;
   var DesignA DesignB Relationship;
run;
title;

                                Candidate Redundant (N-1) Pairings, Least Related First                                 


DesignA  DesignB  Relationship
-------  -------  ------------
G3_T1    G3_T3               0
G3_T2    G3_T4            0.25
G3_T1    G3_T2           0.375
G3_T1    G3_T4           0.375
G3_T2    G3_T3           0.375
G3_T3    G3_T4           0.375

NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Candidate Redundant (N-1) Pairings, Least Related First.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpreting the results

**Inbreeding coefficients (Step 2).** All founding Generation-1 platforms and all Generation-2 derivatives show *F* = 0 — by construction none has two related predecessors. Two Generation-3 models surface with *F* = 0.25: `G3_T1` (built from the sibling pair `G2_D1` x `G2_D2`) and `G3_T3` (from the sibling pair `G2_D3` x `G2_D4`). Their predecessors trace back to the *same* pair of founding platforms, so their heritage is concentrated. From a reliability standpoint these are the designs most exposed to a single inherited defect, and they warrant extra independent qualification testing. `G3_T2` and `G3_T4`, which cross the two disjoint lineages, are *F* = 0.

**Relationship matrix (Step 3).** The off-diagonal entries quantify pairwise shared heritage. The two sibling Generation-2 pairs each show a relationship of 0.50 to one another (`G2_D1`-`G2_D2` and `G2_D3`-`G2_D4`), while derivatives from opposite lineages sit at 0.00. The inbred current-fleet designs carry self-relationships of 1.25 (= 1 + *F*), visible on the diagonal. The `DesignCovar` dataset makes the full matrix available to join against substation assignments.

**Least-related pairings (Step 4).** Ranking every current-fleet pair by relationship puts **`G3_T1` & `G3_T3` first at relationship 0.00** — they descend from entirely disjoint ancestral platforms and share no design heritage. This is the strongest redundant pairing, and it is especially valuable because `G3_T1` is itself one of the two most inbred designs: pairing it with a completely unrelated mate hedges its concentrated heritage. The next-best pair is `G3_T2` & `G3_T4` at 0.25; the remaining pairs sit at 0.375. The fleet's average inbreeding coefficient of **0.0417** (printed by the AVERAGE option in Step 2) summarizes overall design diversity. Procurement should prefer the lowest-relatedness pairs for N-1 positions and, over time, broaden the ancestral base — the asset-engineering equivalent of avoiding a genetic bottleneck.

**Caveat.** This is illustrative synthetic data; a production study would build the pedigree from the manufacturer's real design-revision records and validate the relatedness scores against historical common-mode outage events before driving procurement decisions.